In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import confusion_matrix
#rom mlxtend.plotting import plot_confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

from gensim.models import Word2Vec

2023-03-22 20:34:38.277080: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-03-22 20:34:44.230763: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2023-03-22 20:34:44.230876: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2023-03-22 20:34:44.850492: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-03-22 20:34:56.830335: W tensorflow/stream_executor/platform/de

In [10]:
# Load clickbait classification data
data = pd.read_csv('Processed_data.csv')

# already tokenized
data['tokens'] = data['headline']

# Train the Word2Vec model
w2v_model = Word2Vec(data['tokens'], size=300, window=5, min_count=1, workers=4)
w2v_model.train(data['tokens'],epochs=10,total_examples=len(data['tokens']))

# total numberof extracted words.
vocab=w2v_model.wv.vocab
print("The total number of words are : ",len(vocab))


# Create feature vectors for each document
def feature_vector(tokens):
    feature_vec = np.zeros((100,), dtype='float32')
    num_words = 0
    for word in tokens:
        if word in model:
            num_words += 1
            feature_vec += model[word]
    if num_words != 0:
        feature_vec /= num_words
    return feature_vec

data['features'] = data['tokens'].apply(feature_vector)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data['features'], data['clickbait'], test_size=0.2, random_state=42)

/tmp/ipykernel_49298/2294228036.py:15: DeprecationWarning: Call to deprecated `__contains__` (Method will be removed in 4.0.0, use self.wv.__contains__() instead).
  if word in model:
/tmp/ipykernel_49298/2294228036.py:17: DeprecationWarning: Call to deprecated `__getitem__` (Method will be removed in 4.0.0, use self.wv.__getitem__() instead).
  feature_vec += model[word]


In [12]:
text = data['tokens'].values
labels = data['clickbait'].values
text_train, text_test, y_train, y_test = train_test_split(text, labels)
print(text_train.shape, text_test.shape, y_train.shape, y_test.shape)

(24000,) (8000,) (24000,) (8000,)


In [ ]:
vocab_size = 5000
maxlen = 500
embedding_size = 32


model = Sequential()
model.add(Embedding(vocab_size, embedding_size, input_length=maxlen))
model.add(LSTM(32, return_sequences=True))
model.add(GlobalMaxPooling1D())
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))
model.summary()

In [ ]:
callbacks = [EarlyStopping(monitor='val_accuracy', min_delta=1e-4, patience=3, verbose=1), 
             ModelCheckpoint(filepath='weights.h5', monitor='val_accuracy', mode='max',  save_best_only=True, 
                             save_weights_only=True, verbose=1)]

In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
history = model.fit(X_train, y_train, batch_size=512, validation_data=(x_test, y_test), epochs=20, callbacks=callbacks)

In [2]:
# import pandas as pd
# import numpy as np
# import nltk
# from gensim.models import Word2Vec
# from sklearn.model_selection import train_test_split
# from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
# from sklearn.svm import SVC
# from sklearn.metrics import classification_report
# from sklearn.ensemble import RandomForestClassifier

# # Load clickbait classification data
# data = pd.read_csv('Processed_data.csv')

# # # Tokenize the text data
# # nltk.download('punkt')
# # data['tokens'] = data['headline'].apply(nltk.word_tokenize)

# # Train the Word2Vec model
# #model = Word2Vec(data['tokens'], size=100, window=5, min_count=1, workers=4)

# # Create feature vectors for each document
# def feature_vector(tokens):
#     feature_vec = np.zeros((100,), dtype='float32')
#     num_words = 0
#     for word in tokens:
#         if word in model:
#             num_words += 1
#             feature_vec += model[word]
#     if num_words != 0:
#         feature_vec /= num_words
#     return feature_vec

# data['features'] = data['tokens'].apply(feature_vector)

# # Split the data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(data['features'], data['clickbait'], test_size=0.2, random_state=42)

[nltk_data] Downloading package punkt to /home/pradyumna/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/tmp/ipykernel_49298/746666696.py:26: DeprecationWarning: Call to deprecated `__contains__` (Method will be removed in 4.0.0, use self.wv.__contains__() instead).
  if word in model:
/tmp/ipykernel_49298/746666696.py:28: DeprecationWarning: Call to deprecated `__getitem__` (Method will be removed in 4.0.0, use self.wv.__getitem__() instead).
  feature_vec += model[word]


In [ ]:
# import tensorflow as tf
# from tensorflow.keras.models import Sequential, Model
# from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, Dropout
# vocab_size = 16000
# maxlen = 250
# embedding_size = 64

# model = Sequential()
# model.add(Embedding(vocab_size, embedding_size, input_length=maxlen))
# model.add(LSTM(64))
# model.add(Dense(64, activation='relu'))
# model.add(Dense(1, activation='sigmoid'))
# model.summary()



# model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
# history = model.fit(train_x, y_train, batch_size=512, \
#     validation_data=(X_test, y_test), epochs=3)